<a href="https://colab.research.google.com/github/LionPaul/retro-game-miner/blob/main/snes/notebooks/03_api_populate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
import os
import time

# ==============================================================================
# PIPELINE CONFIGURATION
# ==============================================================================
CONSOLE_NAME = "snes"
CLIENT_ID = ''
CLIENT_SECRET = ''

DATASETS_DIR = "datasets"
# ADJUSTMENT: Strictly following the global numerical tracking pattern
INPUT_FILENAME = f"2_{CONSOLE_NAME}_games_cleaned.xlsx"       # Reading from Code 2
OUTPUT_FILENAME = f"3_{CONSOLE_NAME}_games_populated.xlsx"    # Generating base number 3

INPUT_PATH = os.path.join(DATASETS_DIR, INPUT_FILENAME)
OUTPUT_PATH = os.path.join(DATASETS_DIR, OUTPUT_FILENAME)
# ==============================================================================

def authenticate_igdb(client_id, client_secret):
    """Generates the OAuth2 Access Token required to consume the Twitch IGDB API."""
    url = f"https://id.twitch.tv/oauth2/token?client_id={client_id}&client_secret={client_secret}&grant_type=client_credentials"
    response = requests.post(url)
    response.raise_for_status()
    return response.json()['access_token']

def fetch_score_api(game_name, headers):
    """Executes a textual search on IGDB locked strictly to the SNES platform (ID: 19)."""
    if not game_name or pd.isna(game_name) or str(game_name).strip() == "":
        return None

    url = "https://api.igdb.com/v4/games"
    # Strict filter syntax for SNES platform and limit 1 to save overhead
    query = f'search "{str(game_name).strip()}"; fields name, aggregated_rating, rating; where platforms = (19); limit 1;'

    try:
        response = requests.post(url, headers=headers, data=query)
        if response.status_code == 200:
            data = response.json()
            if data:
                game = data[0]
                # Prioritize critic score (aggregated_rating), fallback to user score (rating)
                score = game.get('aggregated_rating', game.get('rating', None))
                if score:
                    return round(score, 2)
    except Exception:
        pass
    return None

def pipeline_gold_pentatendativa(input_path, output_path):
    """
    Scrapes scores using a 5-step fallback strategy to maximize data coverage.
    Acts as the Gold Populated Layer of the Medallion Architecture.
    """
    print(f"🚀 Starting PENTATENDATIVA API Enrichment for console: {CONSOLE_NAME.upper()}")

    if not os.path.exists(input_path):
        print(f"❌ Error: Silver file '{input_path}' not found. Please run Notebook 02 first.")
        return

    # 1. Load cleaned Silver dataset
    df = pd.read_excel(input_path)
    total_games = len(df)
    print(f"📊 Silver dataset loaded. {total_games} games ready for cascade processing.")

    # 2. Twitch/IGDB Authentication
    print("🔑 Authenticating with IGDB API...")
    try:
        token = authenticate_igdb(CLIENT_ID, CLIENT_SECRET)
        headers_igdb = {
            'Client-ID': CLIENT_ID,
            'Authorization': f'Bearer {token}'
        }
        print("✅ Authentication successful!")
    except Exception as e:
        print(f"❌ Critical authentication failure: {e}")
        return

    # 3. Cascade Search Loop (Pentatendativa)
    final_scores = []
    print(f"\n⚡ Starting massive data mining loop. Monitoring strategies in real-time...\n")

    for idx, row in df.iterrows():
        # Maps the 5 semantic strategies built in the Silver Layer
        strategies = [
            ('T1_Clean', row['T1_Clean']),
            ('T2_Fallback', row['T2_Fallback']),
            ('T3_Keyword', row['T3_Keyword']),
            ('T4_Acronym', row['T4_Acronym']),
            ('T5_NoStopwords', row['T5_NoStopwords'])
        ]

        score = None
        winning_strategy = None

        # Iterates through strategies in order of precision
        for strategy_name, search_term in strategies:
            if pd.notna(search_term):
                score = fetch_score_api(search_term, headers_igdb)
                if score:
                    winning_strategy = strategy_name
                    break # Score found! Breaks loop to prevent unnecessary API requests
                time.sleep(0.1) # Micro-delay between sub-attempts to respect rate limits

        final_scores.append(score)

        # Real-time console tracking logs
        if score:
            print(f"🎯 [{idx+1}/{total_games}] {row['T1_Clean']} -> Score: {score} (Found via: {winning_strategy})")
        else:
            print(f"❌ [{idx+1}/{total_games}] {row['T1_Clean']} -> Not Found in any of the 5 strategies")

        # Mandatory API throttle control (Max 4 requests per second)
        time.sleep(0.26)

    # 4. Data persistence
    df['IGDB_Score'] = final_scores
    df.to_excel(output_path, index=False)

    successful_matches = df['IGDB_Score'].notna().sum()
    success_rate = round((successful_matches / total_games) * 100, 1)

    print(f"\n💾 Gold Populated Layer successfully saved at: {output_path}")
    print(f"📈 FINAL INSIGHTS: {successful_matches} out of {total_games} games populated ({success_rate}% success rate!).")

# Trigger the process
pipeline_gold_pentatendativa(INPUT_PATH, OUTPUT_PATH)